In [4]:
"""
Data exploration notebook for viewing missing values in suicide study dataset.
"""

import pandas as pd
import numpy as np
import sys
from pathlib import Path
from dotenv import load_dotenv
import os

# Load environment variables from the .env file
load_dotenv()

WORKSPACE_PATH = os.getenv("WORKSPACE_PATH")

# Add the parent directory to the system path
sys.path.append(str(WORKSPACE_PATH))

from src.config.config import DATA_DIR

np.random.seed(42)

In [ ]:
output_file_path = DATA_DIR / "imputed"


def analyze_dataframe_columns(dataframe):
    """
    Analyzes DataFrame columns by calculating missing data statistics and unique value counts.

    Args:
        dataframe (pd.DataFrame): Input DataFrame for analysis.

    Returns:
        pd.DataFrame: Analysis results with columns:
            - column_name: Name of each column
            - missing_values_total: Count of missing values
            - missing_values_percent: Percentage of missing values
            - unique_values_count: Count of unique values
            - unique_value_counts: String of unique values and their counts
    """
    results = []

    for col in dataframe.columns:
        missing_total = dataframe[col].isnull().sum()
        missing_percent = 100 * missing_total / len(dataframe)

        unique_count = dataframe[col].nunique()
        value_counts = dataframe[col].value_counts().to_dict()
        value_counts_str = ", ".join([f"{k}: {v}" for k, v in value_counts.items()])

        results.append(
            [col, missing_total, missing_percent, unique_count, value_counts_str]
        )

    analysis_df = pd.DataFrame(
        results,
        columns=[
            "column_name",
            "missing_values_total",
            "missing_values_percent",
            "unique_values_count",
            "unique_value_counts",
        ],
    )

    return analysis_df


def nan_exploration_in_rows(dataframe):
    """
    Analyzes NaN distribution across DataFrame rows.

    Args:
        dataframe (pd.DataFrame): Input DataFrame for analysis.

    Returns:
        pd.DataFrame: Analysis results with columns:
            - NaN_count: Number of NaN values in each row
            - Total: Count of rows with that NaN count
            - Percent: Percentage of rows with that NaN count
    """
    nan_counts = dataframe.isna().sum(axis=1).value_counts()
    full_index = list(range(0, len(dataframe.columns) + 1))
    nan_counts = nan_counts.reindex(full_index, fill_value=0)
    nan_counts = nan_counts.sort_index()
    nan_counts_percent = (nan_counts / len(dataframe)) * 100

    missing_data_rows = pd.concat(
        [
            pd.Series(full_index, name="NaN_count"),
            nan_counts.rename("Total"),
            nan_counts_percent.rename("Percent"),
        ],
        axis=1,
    )

    return missing_data_rows

In [3]:
# Data import
csv_file_path = DATA_DIR / "mapped" / "mapped_data.csv"
df_raw = pd.read_csv(csv_file_path)

In [12]:
# Split data and context
context_columns = [col for col in df_raw.columns if col.startswith("Context_")]
df_data = df_raw.drop(columns=context_columns, inplace=False)

In [13]:
# Data exploration
analysis_df = analyze_dataframe_columns(df_data)
analysis_df

,column_name,missing_values_total,missing_values_percent,unique_values_count,unique_value_counts
0,ID,0,0.000000,128330,"138421942: 1, 126297358,00: 1, 126300391,00: 1..."
1,Date,14518,11.313021,120,"2023-07-01: 1359, 2023-05-01: 1355, 2023-03-01..."
2,AgeGroup,1296,1.009896,16,"19_24: 13764, 30_34: 13377, 35_39: 13287, 25_2..."
3,Gender,7,0.005455,2,"M: 93811, F: 34512"
4,Marital,16270,12.678251,7,"Single: 50741, Married: 40423, Divorced: 8782,..."
5,Education,88152,68.691654,6,"Primary: 12428, Secondary: 12240, Vocational: ..."
6,WorkInfo,62969,49.068028,4,"Employed: 26495, Unemployed: 24645, Student: 1..."
7,Income,55696,43.400608,4,"Steady: 25747, Dependent: 18228, Benefits: 167..."
8,Fatal,71,0.055326,2,"0.0: 68489, 1.0: 59770"
9,Place,107,0.083379,12,"House: 79745, Other: 13810, UtilitySpaces: 116..."


In [14]:
missing_data_rows = nan_exploration_in_rows(df_data)
missing_data_rows

,NaN_count,Total,Percent
0,0,4907,3.823736
1,1,18192,14.175953
2,2,22148,17.258630
3,3,21260,16.566664
4,4,25226,19.657134
5,5,22050,17.182264
6,6,8346,6.503546
7,7,3101,2.416426
8,8,2337,1.821086
9,9,695,0.541573
